[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Muestreo y sesgos de recolección.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo I: *Datos y su calidad*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se distinguirán población, marco, muestra, parámetro y estadístico, se compararán diseños de muestreo por simulación y se revisarán los sesgos de recolección más comunes.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencia de la API: curso py311.

Este curso no repasa la API método por método: esa explicación detallada está en los *notebooks* del curso [py311](https://github.com/PythonistaMX/py311). Aquí se hace un repaso breve y el tiempo se dedica a las **decisiones de análisis**: qué tipo de dato elegir, qué hacer con un faltante y qué consecuencias tiene cada decisión. Para esta sesión:

| Tema | *Notebooks* de py311 |
|---|---|
| *NumPy* | [02 Conceptos básicos de *NumPy*](https://github.com/PythonistaMX/py311/blob/main/02_conceptos_basicos_de_numpy.ipynb) · [03 Gestión de arreglos](https://github.com/PythonistaMX/py311/blob/main/03_gestion_de_arreglos_de_numpy.ipynb) · [04 Arreglos con contenido aleatorio](https://github.com/PythonistaMX/py311/blob/main/04_arreglos_con_contenido_aleatorio.ipynb) · [05 Operaciones básicas con arreglos](https://github.com/PythonistaMX/py311/blob/main/05_operaciones_basicas_con_arreglos.ipynb) |
| *pandas* | [09 Introducción a *pandas*](https://github.com/PythonistaMX/py311/blob/main/09_introduccion_a_pandas.ipynb) · [10 Tipos de datos](https://github.com/PythonistaMX/py311/blob/main/10_tipos_de_datos_de_pandas.ipynb) · [11 Operaciones básicas con *dataframes*](https://github.com/PythonistaMX/py311/blob/main/11_operaciones_basicas_con_dataframes.ipynb) · [12 Índices y multiíndices](https://github.com/PythonistaMX/py311/blob/main/12_indices_y_multiindices.ipynb) · [13 Datos categóricos](https://github.com/PythonistaMX/py311/blob/main/13_datos_categoricos.ipynb) |

> ⚠️ **Nota: py311 usa *pandas* 2.** Sus *notebooks* se escribieron para *pandas* 2.x y este curso usa *pandas* 3, así que algunos resultados se verán distintos. Por ejemplo, allá el texto aparece con `dtype` `object` (aquí es `str`), se explica el parámetro `inplace` y la asignación encadenada como si modificaran el original (con *pandas* 3 ya no lo hacen sobre columnas o selecciones) y puede aparecer `SettingWithCopyWarning`, que ya no existe. La sección 1.1 de `03_pandas_3_esencial.ipynb` explica estos cambios. Cuando haya diferencias, se sigue lo que muestra este *notebook*.

### Preparación del entorno.

Se importarán las bibliotecas con sus alias convencionales, se ubicará la carpeta `data/` (que ya trae los CSV del curso, sin necesidad de internet) y se fijará la semilla `401` para que los resultados aleatorios sean reproducibles.

* La siguiente celda importará las bibliotecas y definirá `DATA` y `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)

* La siguiente celda mostrará las versiones instaladas. El curso usa *pandas* 3.

In [ ]:
{"NumPy": np.__version__, "pandas": pd.__version__, "seaborn": sns.__version__}

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda leerá `penguins.csv`, el *dataset* presentado en `03_pandas_3_esencial.ipynb`.

In [ ]:
pinguinos = pd.read_csv(os.path.join(DATA, "penguins.csv"))
pinguinos.shape

---
## 1. Muestreo y sesgos de recolección (30 min).

Antes de calcular nada hay que preguntar **¿cómo llegaron estos datos aquí?** Casi nunca se mide a toda la población: se mide una **muestra**, y la forma de elegirla determina a quién representan las conclusiones. En el flujo del EDA, esta pregunta se responde junto con el diccionario de datos y condiciona todo lo demás: ningún método estadístico corrige un sesgo de recolección que no se reconoce.

### 1.1 Población, marco, muestra, parámetro y estadístico.

| Concepto | Definición | En `penguins` |
|---|---|---|
| **Población** | Conjunto completo de unidades sobre el que se quiere concluir | Pingüinos adultos reproductores de las tres especies en el archipiélago Palmer |
| **Marco muestral** | Lista o mecanismo con el que se puede **llegar** a las unidades | Nidos de estudio en colonias de tres islas cercanas a la Estación Palmer |
| **Muestra** | Unidades efectivamente medidas | Los 344 pingüinos del *dataset* |
| **Parámetro** | Valor numérico de la población, normalmente desconocido | Masa media $\mu$, proporción de Gentoo $p$ |
| **Estadístico** | Valor calculado con la muestra para **estimar** un parámetro | Media muestral $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$, proporción muestral $\hat{p}$ |

Si el marco no cubre a toda la población (por ejemplo, colonias de otras islas que nunca entran en la lista), hay **error de cobertura**, que ningún tamaño de muestra corrige. La diferencia entre estadístico y parámetro tiene dos componentes: el **error aleatorio** (cambia de muestra a muestra y se reduce al aumentar $n$; se cuantifica en la sesión 5) y el **sesgo** (un error sistemático del diseño, que no se reduce con más datos).

### 1.2 Diseños de muestreo.

En un **muestreo probabilístico** cada unidad del marco tiene una probabilidad **conocida y mayor que cero** de ser elegida; eso permite cuantificar la incertidumbre. En uno **no probabilístico** no se conoce esa probabilidad.

| Diseño | Cómo se elige | Ventajas | Desventajas | Ejemplo |
|---|---|---|---|---|
| **Aleatorio simple** | Cada subconjunto de tamaño $n$ tiene la misma probabilidad | Sencillo; base de casi todas las fórmulas | Requiere un marco completo; puede dejar grupos pequeños subrepresentados por azar | Sortear 60 nidos de la lista completa |
| **Sistemático** | Se elige un inicio al azar y luego una de cada $k = N / n$ unidades | Fácil de aplicar en campo o en una lista | Si la lista tiene un patrón periódico de periodo $k$, se sesga | Medir uno de cada 5 nidos a lo largo de la playa |
| **Estratificado** | Se divide la población en **estratos** homogéneos y se muestrea dentro de cada uno | Garantiza representar a cada grupo; reduce el error si los estratos difieren | Requiere conocer el estrato de cada unidad | Sortear nidos **dentro de cada especie** en proporción a su tamaño |
| **Por conglomerados** | Se sortean **grupos completos** (conglomerados) y se mide todo dentro de ellos | Barato cuando las unidades están agrupadas geográficamente | Más error si los conglomerados difieren entre sí | Sortear colonias (o islas) y medir todos sus nidos |
| **Polietápico** | Combina etapas: se sortean conglomerados y luego unidades dentro de ellos | Práctico en estudios grandes (encuestas nacionales) | Análisis más complejo; hay que ponderar | Sortear islas, luego colonias, luego nidos |

| Diseño no probabilístico | Cómo se elige | Riesgo principal | Ejemplo |
|---|---|---|---|
| **Por conveniencia** | Lo que está más a la mano | Sesgo de selección de dirección desconocida | Medir los pingüinos más cercanos a la estación |
| **Por cuotas** | Se llenan cuotas por grupo, pero sin azar dentro de cada grupo | Dentro de cada cuota se elige por conveniencia | "30 machos y 30 hembras", los primeros que se encuentren |
| **Bola de nieve** | Cada participante recluta a otros | Redes homogéneas; sobrerrepresenta a los muy conectados | Encuestas a poblaciones difíciles de localizar |
| **Intencional (de juicio)** | El investigador elige casos "típicos" o de interés | Depende del criterio de quien elige | Elegir colonias conocidas y accesibles para un estudio de largo plazo |

### 1.3 Simulación: ¿qué tan bien estima cada diseño?

Para comparar diseños hace falta conocer la verdad. Por eso se tratarán los 342 pingüinos con masa registrada como si fueran la **población** completa y se tomarán muestras de 60 con distintos diseños. Se comparará cada muestra con el total en dos parámetros: la **proporción de cada especie** y la **masa media**. Las funciones de *pandas* que se usarán son:

* `df.sample(n=..., random_state=...)`: muestra aleatoria simple sin reemplazo.
* `df.groupby("estrato").sample(frac=..., random_state=...)`: la misma fracción dentro de cada estrato (estratificado **proporcional**).
* `df.iloc[posiciones]`: filas separadas por un paso fijo $k = N / n$ (aquí 342 / 60 = 5.7) a partir de un inicio aleatorio entre 0 y $k$ (sistemático). Con un paso no entero se redondean las posiciones hacia abajo; así la muestra recorre la lista completa y tiene exactamente $n$ filas.

* La siguiente celda definirá la población de juguete y la función `resumen_muestra`, que devuelve la proporción de cada especie, la masa media y el tamaño de una muestra.

In [ ]:
poblacion = pinguinos.dropna(subset=["body_mass_g"]).reset_index(drop=True)
especies = ["Adelie", "Chinstrap", "Gentoo"]

def resumen_muestra(muestra):
    resumen = muestra["species"].value_counts(normalize=True).reindex(especies, fill_value=0).to_dict()
    resumen["masa_media"] = muestra["body_mass_g"].mean()
    resumen["n"] = len(muestra)
    return resumen

N, n = len(poblacion), 60
N, n

* La siguiente celda tomará una muestra con cada diseño (aleatorio simple, estratificado por especie, sistemático, un conglomerado y conveniencia) y comparará sus estimaciones con la población.

In [ ]:
paso = N / n
inicio = rng.uniform(0, paso)
posiciones = (inicio + paso * np.arange(n)).astype(int)
isla_sorteada = str(rng.choice(poblacion["island"].unique()))
muestras = {
    "población": poblacion,
    "aleatorio simple": poblacion.sample(n=n, random_state=401),
    "estratificado por especie": poblacion.groupby("species").sample(frac=n / N, random_state=401),
    "sistemático": poblacion.iloc[posiciones],
    f"conglomerado (isla {isla_sorteada})": poblacion[poblacion["island"] == isla_sorteada],
    "conveniencia (primeras 60 filas)": poblacion.head(n),
}
comparacion = pd.DataFrame({nombre: resumen_muestra(m) for nombre, m in muestras.items()}).T
comparacion.round(3)

**Interpretación:**
* **Aleatorio simple:** se acerca al total, pero por azar quedó con más Gentoo de la cuenta (≈ 42% contra 36%) y por eso sobreestima la masa media en unos 150 g. Es error **aleatorio**: otra semilla daría otra desviación.
* **Estratificado:** las proporciones de especies coinciden con la población **por diseño**, y la masa media queda más cerca. Al fijar la composición, se elimina la parte del error que venía de "qué especies tocaron".
* **Sistemático:** como el archivo está **ordenado por especie e isla**, tomar una fila cada 5.7 recorre todos los grupos y se comporta como un estratificado implícito. Si el orden tuviera un patrón periódico, el mismo método podría sesgarse.
* **Conglomerado:** con un solo conglomerado (una isla) la muestra hereda la composición de esa isla, que no se parece a la total: con esta semilla tocó Torgersen, que solo tiene Adelie, y la masa media queda unos 500 g abajo; Dream no tiene Gentoo y Biscoe no tiene Chinstrap, así que cualquier isla sola sesgaría las proporciones. Por eso un diseño por conglomerados necesita **muchos** conglomerados.
* **Conveniencia:** las primeras filas son todas Adelie de Torgersen; la masa media sale más de 500 g por debajo. Es **sesgo**: tomar más filas "de arriba" no lo corrige.

**Decisión que permite tomar:** cuando se sabe que un grupo (la especie) explica gran parte de la variable de interés (la masa), conviene estratificar por ese grupo. **Error común que se evita:** juzgar un diseño por una sola muestra; la comparación justa es repetir el muestreo muchas veces.

* La siguiente celda repetirá 500 veces el muestreo aleatorio simple y el estratificado y comparará el sesgo y la dispersión de la masa media estimada.

In [ ]:
repeticiones = {"aleatorio simple": [], "estratificado por especie": []}
for _ in range(500):
    repeticiones["aleatorio simple"].append(poblacion.sample(n=n, random_state=rng)["body_mass_g"].mean())
    repeticiones["estratificado por especie"].append(
        poblacion.groupby("species").sample(frac=n / N, random_state=rng)["body_mass_g"].mean())
pd.DataFrame({
    diseno: {"sesgo (g)": np.mean(valores) - poblacion["body_mass_g"].mean(), "desviación estándar (g)": np.std(valores)}
    for diseno, valores in repeticiones.items()
}).T.round(1)

**Interpretación:** los dos diseños son prácticamente **insesgados**: en promedio aciertan con unos pocos gramos de diferencia, muy por debajo de su variabilidad (en el estratificado, una parte de esa diferencia viene de redondear 60 × proporción de cada especie a un número entero de pingüinos: 26, 12 y 22), pero el estratificado varía mucho menos de muestra a muestra: su desviación estándar (unos 52 g) es aproximadamente la mitad que la del aleatorio simple (unos 90 g). Con el mismo costo (60 pingüinos), el estratificado da estimaciones más precisas porque las especies difieren mucho en masa. Esta desviación estándar de un estimador se llama **error estándar** y se estudia a fondo en la sesión 5.

### 1.4 ¿Cómo se recolectó `penguins`?

Según la documentación del *dataset* (Gorman, Williams y Fraser, 2014; *Palmer Station LTER*), se midieron **adultos que anidaban** en colonias de tres islas cercanas a la Estación Palmer, en las temporadas 2007 a 2009. En los datos originales cada individuo se identifica por su **nido** (`N1A1` = nido 1, adulto 1), así que las unidades llegan **agrupadas por nido y por colonia**.

En términos de diseño, es una combinación de elección **intencional** de islas y colonias (accesibles desde la estación y estudiadas a largo plazo) con selección de nidos dentro de ellas: un esquema parecido al **polietápico**, pero sin sorteo en la primera etapa. Consecuencias:
* Las conclusiones valen para esas colonias y esas temporadas; extrapolarlas a "todos los Adelie de la Antártida" sería un error de cobertura.
* La especie y la isla están **confundidas** (Gentoo solo en Biscoe, Chinstrap solo en Dream): las proporciones de especies del *dataset* reflejan dónde se trabajó, no cuántos pingüinos de cada especie existen.
* Los dos adultos de un mismo nido no son independientes entre sí (comparten territorio y alimento), algo que importa en las pruebas de la sesión 6.

**💡 Ideas clave de 4.1 a 4.4:**
* Se distingue población, marco, muestra, parámetro ($\mu$, $p$) y estadístico ($\bar{x}$, $\hat{p}$).
* El error **aleatorio** baja con más datos; el **sesgo** no.
* Estratificar por una variable que explica mucho reduce el error; un solo conglomerado o una muestra por conveniencia pueden sesgar mucho.
* Siempre hay que describir cómo se recolectaron los datos y a qué población representan.

### 1.5 Sesgos de recolección.

El diseño de muestreo es la primera fuente de sesgo, pero no la única. Estos son los sesgos de recolección más frecuentes:

| Sesgo | Qué ocurre | Ejemplo |
|---|---|---|
| **Selección** | La muestra no representa a la población de interés | Medir pingüinos solo en las colonias más accesibles |
| **Supervivencia** | Solo se observa a quienes pasaron un filtro | Analizar solo empresas que siguen operando; aviones que regresaron |
| **No respuesta** | Quienes no responden difieren de quienes sí | Encuesta de ingreso que contestan menos los de ingresos altos |
| **Medición** | El instrumento o el procedimiento distorsiona el valor | Báscula descalibrada; preguntas que inducen la respuesta; redondeos |

**Ejemplo:** se simulará un sesgo de **selección**, equivalente a un muestreo de un solo conglomerado: ¿qué pasaría si solo se hubiera muestreado la isla Biscoe?

* La siguiente celda comparará la proporción de especies en todas las islas y solo en Biscoe.

In [ ]:
completo = pinguinos.dropna(subset=["body_mass_g"])
solo_biscoe = completo[completo["island"] == "Biscoe"]
pd.DataFrame({
    "todas las islas": completo["species"].value_counts(normalize=True),
    "solo Biscoe": solo_biscoe["species"].value_counts(normalize=True),
}).fillna(0).round(3)

* La siguiente celda comparará la masa media en ambos casos.

In [ ]:
{"todas las islas": completo["body_mass_g"].mean().round(0),
 "solo Biscoe": solo_biscoe["body_mass_g"].mean().round(0)}

**Interpretación:** la masa media pasa de 4,202 g a 4,716 g solo por medir en una isla. No es que los pingüinos de Biscoe sean "más pesados" por la isla: Biscoe concentra a los Gentoo, la especie más grande. Es el mismo fenómeno que el conglomerado de la sección 1.3.

La masa media cambia unos 500 g solo por **dónde** se muestreó: Biscoe concentra a los Gentoo, que son los más grandes. La muestra no está mal, pero **las conclusiones solo valen para la población que representa**.

### 🟢 Ejercicio 1 · Identifica el sesgo.

Para cada caso, indica el sesgo principal y una forma de mitigarlo:
1. Una *app* de *fitness* concluye que sus usuarios bajan de peso analizando solo a quienes siguen activos después de 6 meses.
2. Una encuesta telefónica en horario laboral sobre hábitos de consumo.
3. Un estudio de propinas (`tips`) registrado por **un solo mesero** en **un solo restaurante**.
4. Un sensor de temperatura que redondea hacia abajo al entero más cercano.

**🎯 Objetivo:** reconocer el sesgo principal en situaciones reales y proponer cómo mitigarlo. **Qué se espera aprender:** que el sesgo se identifica razonando sobre **cómo se recolectaron** los datos, no mirando los números.

**✍️ Tu respuesta:**

_(escribe aquí)_

**💡 Ideas clave de 4.5:**
* Sesgos de selección, supervivencia, no respuesta y medición: cada uno tiene una mitigación propia.
* Las conclusiones solo valen para la población que la muestra representa.

---
## ✅ Cierre.

**Práctica (sin calificación, en Classroom):** terminar los ejercicios 🟢 pendientes de este *notebook* y, si es posible, un reto 🔴.

**Siguiente *notebook*:** `06_diccionario_de_datos.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>